In [ ]:
import clingo
import os
import pandas as pd
import textwrap
import numpy

In [ ]:
all_ts = []

In [ ]:
# Path to the folder containing the CSV files
folder_path = 'data'
def readDatasInPanda(file_path):
#we probaply should not combine all the files in one dataset just now for testing
    df = pd.read_csv(file_path)
    dataframes = []

        # Append the cleaned DataFrame to the list
    dataframes.append(df)

# Combine all DataFrames into one
    pandas_data = pd.concat(dataframes, ignore_index=True)

# Display the first few rows of the combined DataFrame
    return pandas_data
    #voltages_over_time = combined_data['ex1.r_load.v']
# Save the combined DataFrame to a new CSV file (optional)
#combined_data.to_csv('combined_data.csv', index=False)

In [ ]:
DOMAIN_ASP = textwrap.dedent("""% Constants

% ********************
% 2. Define Entities
% ********************

% Batteries
battery(b1). 
battery(b2). 
battery(b3).

% Switches
switch(switch1). 
switch(switch2). 
switch(switch3).

% ********************
% 3. Define Connections
% ********************

% Each switch is connected to a specific battery
connected(switch1, b1).
connected(switch2, b2).
connected(switch3, b3).


% ********************
% 9. Fault Detection
% ******************** 

battery_fault(B, T) :-
    total_voltage(null, T),
    connected(S, B),
    switch_state(S, on, T),
    time(T).

switch_fault(S, T) :-
    total_voltage(low,T),
    switch_state(S,on,T).
    

""")

In [ ]:
def classify_voltage(value: float, target: float, percentage: float, tolerance: float) -> str:
    """
    Map a real-valued voltage to one of the qualitative labels:
      vlow < (vmid,vlow) < vmid < (vmax,vmid) < vmax
    """

    target = target * (percentage / 100)
   # return int(value)
    if value < 0.5:
        return "null"
    elif value >= target * tolerance and value <= target * (2.0 - tolerance):
        return "target"
    elif value > target * (2.0 - tolerance):
        return "high"
    elif value < target * tolerance:
        return "low"


In [ ]:
def build_facts_from_csv(pandasData) -> str:
    """
    Reads time, Vbattery1, Vbattery2, Vbattery3, Vout, switch1, switch2, switch3 from a CSV file,
    maps Vout to qualitative states, and returns ASP facts as a string.
    """
    data = []
    all_ts.clear()

   

    for index, row in pandasData.iterrows():
        lines = []
        # Extract and classify output voltage
        t = index  # Convert time to integer

        #what you see here are python shenanigans
        ts = row["time"]
        py_ts = ts.item()
        new_py_ts = str(py_ts)
        all_ts.append(new_py_ts)

        
        vout_value = "Nan"
        vout_value = float(row["circ1.r_load.v"])  # Extract voltage value     
        # Classify Vout into qualitative state
        vout_state = classify_voltage(vout_value, float(row["circ1.target_v"]), float(row["circ1.r_load.percentage"]), float(row["circ1.tolerance"]))
        
        # Append time and voltage facts
        lines.append(f"time({t}).")
        lines.append(f"total_voltage({vout_state},{t}).")
        # Extract and process switch states
        switch1_state = row["circ1.s[1].mode"]
        switch2_state = row["circ1.s[2].mode"]
        switch3_state = row["circ1.s[3].mode"]  # Corrected key

        
        # Ensure switch states are either 'on' or 'off'
        #print( str(switch1_state) == "1.0")
        #print(switch1_state)
        switch1 = "on" if str(switch1_state) == "2.0" else "off"
        switch2 = "on" if str(switch2_state) == "2.0" else "off"
        switch3 = "on" if str(switch3_state) == "2.0" else "off"

        # Append switchState facts
        lines.append(f"switch_state(switch1,{switch1},{t}).")
        lines.append(f"switch_state(switch2,{switch2},{t}).")
        lines.append(f"switch_state(switch3,{switch3},{t}).")

        data.append(lines)
        #if index == 100: 
          #  break
   # for i in data:
    #    print(i)
    return data

In [ ]:
def on_model(model, output):
    """
    Callback function to process each model found by Clingo.
    Collects the voltage assignments and any detected faults.
    """
    atoms = model.symbols(shown=True)
    battery_assignments = [str(a) for a in atoms if a.name == "voltage"]
   # heater_assignments = [str(a) for a in atoms if a.name == "heaterVoltage"]
    faults = [str(a) for a in atoms if a.name == "faulty"]

    output.append((battery_assignments, faults))

In [ ]:
def live_diagnose_simulation(alls_data_facts, time_range):
    counter_max = len(alls_data_facts) - 1
    counter = 1

    asp_data = []
    temp = []
    for facts in alls_data_facts:
        s = "\n".join(facts)
        temp.append(s)
        #print(s)

    asp_data = "\n".join(temp)
    #print(asp_data)

    program = f"{DOMAIN_ASP}\n\n% Observed Data:\n{asp_data}\n"
        
    # 4) Optionally, print the complete program for debugging
    write_to_file = True
    if write_to_file:
        #print("Complete ASP Program:\n")
        try:
            with open("diagnosis_program.asp", "w") as asp_file:
                asp_file.write(program)
           # print(f"ASP program written to '{"diagnosis_program.asp"}'.")
        except IOError as e:
            print(f"Error writing ASP program to file: {e}")
            return
   
    # 3) Initialize Clingo Control
    ctl = clingo.Control()

    # 4) Add the program
    ctl.add("base", [], program)
    ctl.ground([("base", [])])
   #+ print("Finished Init clingo\n")
    # 5) Prepare to collect solutions
    solutions = []

    # 6) Define the callback
    def callback(model):
        battery_assignments = []
        faults = []
        for atom in model.symbols(shown=True):
            if atom.name == "total_voltage":
                battery_assignments.append(str(atom))
            elif atom.name == "battery_fault" or atom.name == "switch_fault" or atom.name == "voltage_fault":
                ts = "Timestamp: " + str(all_ts[atom.arguments[1].number])
                faults.append(ts)
                faults.append(str(atom))
        solutions.append((battery_assignments, faults))

    # 7) Solve
    ctl.solve(on_model=callback)


    if not solutions:
        print("No solutions found. Possible multiple faults or inconsistent data.")
    else:
        for idx, (bats, faults) in enumerate(solutions, start=1):
            if faults:
                for fault in faults:
                    print(f"  {fault}")
        

    
  
        

In [ ]:
def diagnose(csv_path: str, asp_output_path: str = "diagnosis_program.asp"):
    print("Finished reading data\n") 
    data_facts = build_facts_from_csv(readDatasInPanda(csv_path))
    live_diagnose_simulation(data_facts, 0)
      

In [ ]:
diagnose("./data/Circuit_ScenarioVoltageDropsBat_res.csv")